In [2]:
import pandas as pd
import numpy as np

In [ ]:
class Lda:
    def fit(self,X,y):
        self.classes = np.unique(y)
        samples, features = X.shape
        self.means = {}
        self.prior = {}
        for c in self.classes:
            Xc = X[y == c]
            self.means[c] = Xc.mean(axis=0)
            self.prior[c] = Xc.shape[0]/samples
        cov = np.zeros((features,features))
        for c in self.classes:
            Xc = X[y == c]
            centered = Xc - self.means[c]
            cov += centered.T @ centered
        cov /= (samples - len(self.classes))
        self.cov = cov
        self.cov_inv = np.linalg.inv(cov)
        return self
    
    def predict(self,X):
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                mu = self.means[c]
                scores[c] = x @ self.cov_inv @ mu - 0.5 * mu @ self.cov_inv @ mu + np.log(self.prior[c])
            preds.append(max(scores,key=scores.get)) 
        return np.array(preds)
        

In [6]:
class Qda:
    def fit(self,X,y):
        self.classes = np.unique(y)
        samples, parameters = X.shape
        self.means = {}
        self.priors = {}
        self.cov = {}
        self.cov_inv = {}
        self.logdets = {}
        for c in self.classes:
            Xc = X[y == c]
            self.means[c] = Xc.mean(axis=0)
            self.priors[c] = Xc.shape[0]/samples

            centered = Xc - self.means[c]
            cov = (centered.T @ centered) / (Xc.shape[0] - 1)
            cov += np.eye(X.shape[1])* 1e-6
            self.cov[c] = cov
            self.cov_inv[c] = np.linalg.inv(cov)
            sign,logdet = np.linalg.slogdet(cov)
            self.logdets[c] = logdet
        return self
    def predict(self,X):
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                mu = self.means[c]
                diff = x - mu
                scores[c] = -0.5 * self.logdets[c] - 0.5 * diff @ self.cov_inv[c] @ diff + np.log(self.priors[c])
            preds.append(max(scores, key=scores.get))
        return np.array(preds)
    